In [2]:
import pandas as pd
import numpy as np
import sys
from xgboost import XGBClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.metrics import classification_report, f1_score, hamming_loss, accuracy_score
import time
from pathlib import Path
import mlflow
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
pd.set_option('display.max_rows', None)

# ==========================================
# 0. MLFLOW SETUP
# ==========================================
root_path = Path.cwd().parent
mlflow.set_tracking_uri((root_path / "mlruns").as_uri())
mlflow.set_experiment("07e_Baseline_XGBoost_AllFeatures")

# ==========================================
# 1. LOAD DATA DARI FOLDER YANG BENAR
# ==========================================
print("🔍 1. Memuat dataset...")
train_balanced = pd.read_csv(root_path / "Data/processed/train_balanced_multilabel.csv")
test_data      = pd.read_csv(root_path / "Data/split/test_data.csv")

# ==========================================
# 2. PISAHKAN FITUR (X) DAN TARGET (y)
# ==========================================
target_cols = ['risk_depression', 'risk_anxiety', 'risk_stress']

X_train = train_balanced.drop(columns=target_cols)
y_train = train_balanced[target_cols].astype(int)  # ← Fix: pastikan integer

X_test  = test_data.drop(columns=target_cols)
y_test  = test_data[target_cols].astype(int)

# ✅ Samakan kolom test mengikuti train (jaga-jaga ada kolom beda)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

num_features = X_train.shape[1]
print(f"✓ Jumlah fitur (SEMUA - Tanpa Seleksi): {num_features} fitur")
print(f"✓ Data Train shape: {X_train.shape}")
print(f"✓ Data Test shape:  {X_test.shape}")
print(f"✓ Kolom sama: {X_train.shape[1] == X_test.shape[1]}")

# ==========================================
# 3. HELPER FUNCTIONS - THRESHOLD OPTIMIZATION
# ==========================================
def find_optimal_thresholds(model, X_train, Y_train, targets, step=0.01):
    """
    Mencari threshold optimal per target class menggunakan training data.
    Menggunakan predict_proba dari ClassifierChain secara langsung
    untuk menghindari feature shape mismatch.
    """
    print("\n   Mengambil probabilitas dari training data...")
    
    # ✅ Gunakan decision_function dari chain langsung
    # Ambil proba dengan cara manual per estimator menggunakan X_aug yang benar
    n_samples  = X_train.shape[0]
    n_targets  = len(targets)
    all_probas = np.zeros((n_samples, n_targets))

    # Rekonstruksi augmented features seperti yang dilakukan ClassifierChain
    X_aug = X_train.values.copy() if hasattr(X_train, 'values') else X_train.copy()

    for i, estimator in enumerate(model.estimators_):
        # Augmented X sudah berisi prediksi label sebelumnya
        # Ukuran X_aug bertambah setiap iterasi (itulah kenapa harus rekonstruksi)
        proba     = estimator.predict_proba(X_aug)[:, 1]
        all_probas[:, i] = proba

        # Tambahkan prediksi label ini ke X_aug untuk chain berikutnya
        pred_label = (proba >= 0.5).astype(int).reshape(-1, 1)
        X_aug      = np.hstack([X_aug, pred_label])

    # Cari threshold terbaik per target
    best_thresholds = []
    print(f"\n   {'Target':<20} {'Threshold':>10} {'Train F1':>10}")
    print(f"   {'-'*42}")

    for i, target_name in enumerate(targets):
        best_t  = 0.5
        best_f1 = 0.0

        for t in np.arange(0.1, 0.9, step):
            y_pred_temp = (all_probas[:, i] >= t).astype(int)
            f1 = f1_score(
                Y_train.iloc[:, i] if hasattr(Y_train, 'iloc') else Y_train[:, i],
                y_pred_temp,
                zero_division=0
            )
            if f1 > best_f1:
                best_f1 = f1
                best_t  = t

        best_thresholds.append(round(best_t, 2))
        print(f"   {target_name:<20} {best_t:>10.2f} {best_f1:>10.4f}")

    return best_thresholds


def apply_thresholds(model, X_test, thresholds, targets):
    """
    Menerapkan threshold kustom ke prediksi test data.
    Rekonstruksi augmented features secara manual agar konsisten.
    """
    n_samples  = X_test.shape[0]
    n_targets  = len(targets)
    all_probas = np.zeros((n_samples, n_targets))

    X_aug = X_test.values.copy() if hasattr(X_test, 'values') else X_test.copy()

    for i, estimator in enumerate(model.estimators_):
        proba = estimator.predict_proba(X_aug)[:, 1]
        all_probas[:, i] = proba

        # Gunakan threshold optimal saat augmentasi chain
        pred_label = (proba >= thresholds[i]).astype(int).reshape(-1, 1)
        X_aug      = np.hstack([X_aug, pred_label])

    # Terapkan threshold ke semua prediksi
    y_pred_optimized = np.zeros((n_samples, n_targets), dtype=int)
    for i in range(n_targets):
        y_pred_optimized[:, i] = (all_probas[:, i] >= thresholds[i]).astype(int)

    return y_pred_optimized


def evaluate_predictions(y_true, y_pred, label=""):
    """Hitung semua metrik evaluasi."""
    macro_f1    = f1_score(y_true, y_pred, average='macro', zero_division=0)
    hamming     = hamming_loss(y_true, y_pred)
    exact_match = accuracy_score(y_true, y_pred)
    return {
        "label":       label,
        "Macro_F1":    macro_f1,
        "Hamming_Loss": hamming,
        "Exact_Match": exact_match,
    }

# ==========================================
# 4. INISIALISASI MODEL
# ==========================================
xgb_params = {
    'n_estimators':    300,
    'learning_rate':   0.05,
    'max_depth':       3,
    'gamma':           0.2,
    'colsample_bytree': 0.7,
    'random_state':    42,
    'eval_metric':     'logloss',   # ← Fix: paksa mode classifier
    'n_jobs':          -1
}

print("\n🔧 2. Inisialisasi XGBoost dengan parameter standardized:")
for k, v in xgb_params.items():
    print(f"   - {k}: {v}")

base_xgb    = XGBClassifier(**xgb_params)
chain_model = ClassifierChain(base_xgb, order='random', random_state=42)

# ==========================================
# 5. TRAINING
# ==========================================
print(f"\n⏳ 3. Memulai training ({num_features} ALL Features)...")
start_time    = time.time()
chain_model.fit(X_train, y_train)
training_time = time.time() - start_time
print(f"   ✓ Training selesai dalam {training_time:.2f} detik.")

# ==========================================
# 6. EVALUASI DEFAULT (THRESHOLD 0.5)
# ==========================================
print("\n📊 4. Prediksi dengan threshold default (0.5)...")
y_pred_default  = chain_model.predict(X_test)
results_default = evaluate_predictions(y_test, y_pred_default, "Default (0.5)")

# ==========================================
# 7. THRESHOLD OPTIMIZATION
# ==========================================
print("\n🔍 5. Mencari threshold optimal...")
optimal_thresholds = find_optimal_thresholds(chain_model, X_train, y_train, target_cols)

print(f"\n   Threshold optimal: {dict(zip(target_cols, optimal_thresholds))}")

# ==========================================
# 8. EVALUASI DENGAN THRESHOLD OPTIMAL
# ==========================================
print("\n📊 6. Prediksi dengan threshold optimal...")
y_pred_optimized  = apply_thresholds(chain_model, X_test, optimal_thresholds, target_cols)
results_optimized = evaluate_predictions(y_test, y_pred_optimized, "Optimized")

# ==========================================
# 9. TAMPILKAN HASIL PERBANDINGAN
# ==========================================
print("\n" + "="*70)
print("🏆 HASIL: XGBOOST ALL FEATURES + THRESHOLD OPTIMIZATION")
print("="*70)
print(f"{'Metrik':<25} {'Default (0.5)':>15} {'Optimized':>15} {'Δ':>10}")
print("-"*70)

for metric in ['Macro_F1', 'Hamming_Loss', 'Exact_Match']:
    val_def  = results_default[metric]
    val_opt  = results_optimized[metric]
    delta    = val_opt - val_def
    arrow    = "↑" if delta > 0 else ("↓" if delta < 0 else "→")
    print(f"{metric:<25} {val_def:>15.4f} {val_opt:>15.4f} {arrow} {delta:>+.4f}")

print("="*70)
print(f"\nJumlah Fitur   : {num_features}")
print(f"Training Time  : {training_time:.2f} detik")
print(f"Threshold Optimal:")
for col, t in zip(target_cols, optimal_thresholds):
    print(f"   {col:<25}: {t:.2f}")

print("\n📋 CLASSIFICATION REPORT (Threshold Optimal):")
print(classification_report(
    y_test, y_pred_optimized,
    target_names=['Depression', 'Anxiety', 'Stress']
))

sys.stdout.flush()
# ==========================================
# 10. MLFLOW TRACKING
# ==========================================
print("\n📈 7. Tracking ke MLflow...")
with mlflow.start_run(run_name="XGBoost_AllFeatures_ThresholdOptimized"):
    mlflow.log_params(xgb_params)
    mlflow.log_param("feature_selection_method", "None - All Features")
    mlflow.log_param("multilabel_strategy", "ClassifierChain")
    mlflow.log_param("num_features", num_features)

    # Metrik default
    mlflow.log_metric("Default_Macro_F1",    results_default['Macro_F1'])
    mlflow.log_metric("Default_Hamming_Loss", results_default['Hamming_Loss'])
    mlflow.log_metric("Default_Exact_Match", results_default['Exact_Match'])

    # Metrik setelah optimasi
    mlflow.log_metric("Optimized_Macro_F1",    results_optimized['Macro_F1'])
    mlflow.log_metric("Optimized_Hamming_Loss", results_optimized['Hamming_Loss'])
    mlflow.log_metric("Optimized_Exact_Match", results_optimized['Exact_Match'])

    # Threshold optimal
    for col, t in zip(target_cols, optimal_thresholds):
        mlflow.log_metric(f"Threshold_{col}", t)

    mlflow.log_metric("Training_Time_seconds", training_time)
    mlflow.log_dict(
        {"optimal_thresholds": dict(zip(target_cols, optimal_thresholds))},
        "07e_optimal_thresholds.json"
    )
    print("   ✓ MLflow tracking selesai!")

print("\n✅ Notebook eksekusi SELESAI!")

🔍 1. Memuat dataset...
✓ Jumlah fitur (SEMUA - Tanpa Seleksi): 37 fitur
✓ Data Train shape: (8724, 37)
✓ Data Test shape:  (5436, 37)
✓ Kolom sama: True

🔧 2. Inisialisasi XGBoost dengan parameter standardized:
   - n_estimators: 300
   - learning_rate: 0.05
   - max_depth: 3
   - gamma: 0.2
   - colsample_bytree: 0.7
   - random_state: 42
   - eval_metric: logloss
   - n_jobs: -1

⏳ 3. Memulai training (37 ALL Features)...
   ✓ Training selesai dalam 1.36 detik.

📊 4. Prediksi dengan threshold default (0.5)...

🔍 5. Mencari threshold optimal...

   Mengambil probabilitas dari training data...

   Target                Threshold   Train F1
   ------------------------------------------
   risk_depression            0.42     0.7910
   risk_anxiety               0.18     0.7847
   risk_stress                0.69     0.7773

   Threshold optimal: {'risk_depression': np.float64(0.42), 'risk_anxiety': np.float64(0.18), 'risk_stress': np.float64(0.69)}

📊 6. Prediksi dengan threshold optimal.

d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average

   ✓ MLflow tracking selesai!

✅ Notebook eksekusi SELESAI!
